In [1]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.get_device_name(0))  #

True
NVIDIA GeForce RTX 3080 Ti


In [3]:
import os
from dotenv import load_dotenv

model_id_base = "meta-llama/Meta-Llama-3.1-8B" # General model

load_dotenv()

# Access variables
token = os.getenv("hf_token")

In [2]:
import datasets
from datasets import load_dataset

dataset = load_dataset('json', data_files='data/krusty_krab.jsonl')

c:\Users\Brayden Turner\Projects\chat_playground\chat_playground\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 62923 examples [00:00, 805975.20 examples/s]


In [4]:
from transformers import AutoTokenizer, LlamaForCausalLM, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch
compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, # 8 bit uses too much memory
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True
        )
model_id_base = "meta-llama/Meta-Llama-3.1-8B" # General model


# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id_base)

# Load the model with quantization and auto device mapping
model = AutoModelForCausalLM.from_pretrained(
    model_id_base,
    quantization_config=bnb_config,
    device_map="auto"  # Automatically uses GPU, with CPU fallback if necessary
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:22<00:00,  5.71s/it]


In [7]:
dataset["train"][3]

{'index': 82751,
 'text': 'Conner isn’t responding',
 'date_utc': 1571749983000,
 'is_from_me': 0,
 'cache_has_attachments': 0,
 'message_id': 104714,
 'reaction': None,
 'mime_type': None,
 'name': 'Erik Tharp',
 'members': 'Erik Tharp,Conner Morton,Cole Lewis'}

In [ ]:
def preprocess_long_chat(chat_history, window_size=5, overlap=2):
    chunks = []
    for i in range(0, len(chat_history), window_size - overlap):
        chunk = chat_history[i:i + window_size]
        if len(chunk) > 1:
            chunks.append({"messages": chunk})
    return chunks
